In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# ===============================
# PATHS
# ===============================
ml_base = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
de_base = "/n/scratch/users/a/adm808/Revision/Clincal_batch_DE_Outputs_revision"
pt_path = "/n/groups/patel/adithya/Pseudotime_Outputs_HVG_final/publication_summary_statistics.csv"

cell_types = ["Ast", "Mic", "In", "Oli", "Opc", "Ex"]

de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In":  "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex":  "poisson_DE_results_Ex.csv"
}

# ===============================
# 1. ML PREDICTORS
# ===============================
ml_genes = defaultdict(set)

for ct in cell_types:
    gene_counts = defaultdict(int)

    for split in range(1, 6):
        path = os.path.join(ml_base, ct, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(path):
            continue

        model = joblib.load(path)
        for g, imp in zip(model.feature_names_in_, model.feature_importances_):
            if imp > 0:
                gene_counts[g] += 1

    for g, c in gene_counts.items():
        if c >= 2:
            ml_genes[ct].add(g)

# ===============================
# 2. DEG GENES
# ===============================
deg_genes = {}

for ct in cell_types:
    df = pd.read_csv(os.path.join(de_base, de_files[ct]))
    df = df[(df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)]
    deg_genes[ct] = set(df["gene"])

# ===============================
# 3. PSEUDOTIME GENES (TOP 20, AGGREGATED BY CELL TYPE)
# ===============================
pt_df = pd.read_csv(pt_path)

pt_df["top_trajectory_genes"] = pt_df["top_trajectory_genes"].fillna("").apply(
    lambda x: [g.strip() for g in x.split(",")]
)

pt_genes = defaultdict(set)

def infer_cell_type_from_subcluster(subcluster):
    for ct in cell_types:
        if subcluster.startswith(ct):
            return ct
    return None

for _, row in pt_df.iterrows():
    subcluster = row["Subcluster"]
    ct = infer_cell_type_from_subcluster(subcluster)
    if ct is None:
        continue

    top20 = row["top_trajectory_genes"][:20]
    pt_genes[ct].update(top20)

# ===============================
# TRI-SUPPORTED GENES
# ===============================
tri_supported = {}



print("\n=== TRI-SUPPORTED GENE COUNTS ===")
for ct in cell_types:
    tri = ml_genes[ct] & deg_genes[ct] & pt_genes[ct]
    tri_supported[ct] = tri
    print(f"{ct}: {len(tri)} genes")

# Clean gene name types
for ct in tri_supported:
    tri_supported[ct] = set(str(g) for g in tri_supported[ct])

print("\n=== TRI-SUPPORTED GENES (PER CELL TYPE) ===")
for ct, genes in tri_supported.items():
    print(f"\n{ct}:")
    print(sorted(genes))


=== TRI-SUPPORTED GENE COUNTS ===
Ast: 4 genes
Mic: 2 genes
In: 2 genes
Oli: 6 genes
Opc: 2 genes
Ex: 4 genes

=== TRI-SUPPORTED GENES (PER CELL TYPE) ===

Ast:
['GFAP', 'MT2A', 'MT3', 'PRKG1']

Mic:
['HS3ST4', 'SYTL3']

In:
[np.str_('NRG1'), np.str_('RASGEF1B')]

Oli:
['APOD', 'EIF1', 'MID1IP1', 'PPP2R2B', 'QDPR', 'SPP1']

Opc:
['APOD', 'OLIG1']

Ex:
[np.str_('CLSTN2'), np.str_('LINGO1'), np.str_('RASGEF1B'), np.str_('SLC26A3')]


In [4]:
# Clean gene name types
for ct in tri_supported:
    tri_supported[ct] = set(str(g) for g in tri_supported[ct])

print("\n=== TRI-SUPPORTED GENES (PER CELL TYPE) ===")
for ct, genes in tri_supported.items():
    print(f"\n{ct}:")
    print(sorted(genes))


=== TRI-SUPPORTED GENES (PER CELL TYPE) ===

Ast:
['GFAP', 'MT2A', 'MT3', 'PRKG1']

Mic:
['HS3ST4', 'SYTL3']

In:
['NRG1', 'RASGEF1B']

Oli:
['APOD', 'EIF1', 'MID1IP1', 'PPP2R2B', 'QDPR', 'SPP1']

Opc:
['APOD', 'OLIG1']

Ex:
['CLSTN2', 'LINGO1', 'RASGEF1B', 'SLC26A3']
